# QLoRA Fine-Tuning of Gemma-2 with SFT and GRPO

---

Fine-tuning **`google/gemma-2-2b-it`** with Parameter-Efficient Fine-Tuning (PEFT) for two tasks:

| Part | Objective | Technique |
|------|-----------|-----------|
| **A** | Yoda-style text generation | QLoRA + SFT |
| **B** | Mathematical reasoning with Yoda style | GRPO + composite reward |

**Hardware:** NVIDIA A100-SXM4-40GB &nbsp;·&nbsp; **Framework:** HuggingFace TRL + PEFT  
**Repository:** https://github.com/camilletyriard-dev/gemma2-qlora-sft-grpo  
**Weights:** 🤗 https://huggingface.co/camilletyriard/gemma2-qlora-sft-grpo

---

## 0 · Setup

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes
!pip install -q wandb matplotlib pandas seaborn scikit-learn sentencepiece python-dotenv spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import gc, os, re, json, random, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import wandb
from datasets import load_dataset, load_from_disk, DatasetDict, Dataset as HFDataset
from transformers import (
    AutoModelForCausalLM, AutoModelForSequenceClassification,
    AutoTokenizer, Trainer, DataCollatorWithPadding,
)
from peft import LoraConfig, TaskType, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer, GRPOConfig, GRPOTrainer
from huggingface_hub import login

warnings.filterwarnings("ignore")

In [ ]:
# ── Credentials — set as environment variables, never hardcode ──────────────
# In Google Colab:
#   from google.colab import userdata
#   os.environ["HF_TOKEN"]       = userdata.get("HF_TOKEN")
#   os.environ["WANDB_API_KEY"]  = userdata.get("WANDB_API_KEY")
#   os.environ["CHECKPOINT_DIR"] = "/content/drive/My Drive/checkpoints"

login(os.environ.get("HF_TOKEN", ""))
wandb.login(key=os.environ.get("WANDB_API_KEY", ""))

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # not in Colab

BASE_DIR = Path(os.environ.get("CHECKPOINT_DIR", "./checkpoints"))
BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint directory: {BASE_DIR}")

In [ ]:
# ── Global configuration ─────────────────────────────────────────────────────
import sys; sys.path.insert(0, "..")   # make src/ importable from notebooks/

from src.config import PATHS, SEED, BASE_MODEL_NAME, YODA_DATASET_NAME, QA_DATASET_NAME, REASONING_DATASET_NAME, CLASSIFIER_MODEL_NAME

# Override PATHS to use BASE_DIR from environment
PATHS = {
    "base_model":             BASE_MODEL_NAME,
    "sft_yoda":               BASE_DIR / "sft_yoda",
    "sft_yoda_answ":          BASE_DIR / "sft_yoda_answ",
    "rl_yoda_answ_from_sft":  BASE_DIR / "rl_yoda_answ_from_sft",
    "rl_yoda_answ_from_base": BASE_DIR / "rl_yoda_answ_from_base",
    "synthetic_qa":           BASE_DIR / "synthetic_yoda_qa",
    "classifier_data":        BASE_DIR / "classifier_dataset",
    "classifier_yoda":        BASE_DIR / "classifier_yoda",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Import all utilities from src/ ────────────────────────────────────────────
from src.model import (
    load_base_model_and_tokenizer, load_peft_model,
    checkpoint_exists, get_vram_usage, free_memory,
)
from src.generation import (
    format_prompt_yoda, format_prompt_gsm8k, format_prompt_qa,
    format_prompt_and_answer_qa, generate_response,
    generate_batch_responses, split_sentences, display_examples,
)
from src.data.yoda import load_yoda_dataset, format_yoda_translation_example, format_qa_yoda_example
from src.data.gsm8k import load_gsm8k_dataset, prepare_rl_dataset, extract_gsm8k_answer_text
from src.data.qa_filter import QAFilter
from src.training.sft import get_lora_config, build_sft_trainer
from src.training.grpo import get_grpo_lora_config, build_grpo_trainer
from src.training.classifier import build_classifier_dataset, build_classifier_trainer
from src.rewards.correctness import correctness_reward, correctness_reward_single, extract_gsm8k_final_answer
from src.rewards.format import format_reward
from src.rewards.style import load_style_classifier, style_reward
from src.evaluation.metrics import evaluate_classifier, score_response, compare_before_after
from src.evaluation.plotting import (
    plot_training_loss, plot_reward_curves,
    plot_experiment_comparison,
)

def set_seed(seed: int = SEED) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()
print("All modules loaded.")

---

## Part A · SFT for Yoda-Style Generation

Three-stage pipeline:
1. **A-1** — Baseline inference: establish zero-shot Yoda translation ability
2. **A-2** — SFT: fine-tune Gemma-2 to translate English → Yoda
3. **A-3** — Synthetic dataset: generate Yoda-style QA pairs at scale
4. **A-4** — SFT: train Gemma-2 to answer questions in Yoda style

### A-1 · Baseline Inference

Load the base model (8-bit) and test zero-shot Yoda translation. Reports VRAM usage and three translation examples.

In [ ]:
base_model, tokenizer = load_base_model_and_tokenizer(
    model_name=PATHS["base_model"], quantize=True, quantize_n_bits=8
)
print(f"Model loaded. VRAM: {get_vram_usage()}")

In [ ]:
BASELINE_PROMPTS = [
    "The Force is strong inside this cave.",
    "The organization strived for innovation through creating a transparent structure.",
    "Paris is a city renowned for its art galleries.",
]

print(f"VRAM before inference: {get_vram_usage()}")
for i, prompt in enumerate(BASELINE_PROMPTS, 1):
    response = generate_response(base_model, tokenizer, format_prompt_yoda(prompt, tokenizer))
    print(f"\nExample {i}:")
    print(f"  Input:  {prompt}")
    print(f"  Output: {response}")
print(f"\nVRAM after inference: {get_vram_usage()}")

### A-2 · SFT: English → Yoda Translation

Fine-tune with LoRA on `dvgodoy/yoda_sentences`. Dataset: 648 train / 72 validation.

In [ ]:
yoda_ds = load_yoda_dataset()
print(f"Train: {len(yoda_ds['train'])}  |  Val: {len(yoda_ds['test'])}")
print("\nSample formatted example:")
sample = format_yoda_translation_example(yoda_ds["train"][0], tokenizer)
print(sample["text"][:300])

In [ ]:
# ── Token length analysis ────────────────────────────────────────────────────
sample_texts = [format_yoda_translation_example(ex, tokenizer)["text"]
                for ex in yoda_ds["train"]]
lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in sample_texts]

df_len = pd.DataFrame({"Token Length": lengths})
print(df_len.describe().T.round(1))

plt.figure(figsize=(8, 3))
sns.histplot(lengths, bins=30, color="#4C72B0", edgecolor="white")
plt.axvline(np.mean(lengths), color="#DD8452", lw=2, label=f"Mean: {np.mean(lengths):.0f}")
plt.title("Token length distribution — Yoda translation dataset", weight="bold")
plt.xlabel("Tokens"); plt.ylabel("Count"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Train or load SFT translator ─────────────────────────────────────────────
skip_training_a2 = True   # set False to retrain

if checkpoint_exists(PATHS["sft_yoda"]) and skip_training_a2:
    print("Loading SFT translator from checkpoint...")
    sft_translator_model, tokenizer = load_peft_model(PATHS["sft_yoda"])
    sft_yoda_log_history = json.load(
        open(PATHS["sft_yoda"] / "trainer_state.json")
    )["log_history"]
else:
    print("Training SFT translator...")
    model, tokenizer = load_base_model_and_tokenizer(quantize=True, quantize_n_bits=4)

    train_ds = yoda_ds["train"].map(lambda ex: format_yoda_translation_example(ex, tokenizer))
    val_ds   = yoda_ds["test"].map(lambda ex: format_yoda_translation_example(ex, tokenizer))

    lora_cfg = get_lora_config(r=4, lora_alpha=8, lora_dropout=0.05)
    trainer  = build_sft_trainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_ds, eval_dataset=val_ds,
        output_dir=PATHS["sft_yoda"],
        num_train_epochs=3, per_device_train_batch_size=4,
        learning_rate=2e-4, lora_config=lora_cfg,
    )
    trainer.train()
    trainer.save_model(str(PATHS["sft_yoda"]))
    sft_yoda_log_history = trainer.state.log_history
    sft_translator_model = trainer.model

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────────
plot_training_loss(sft_yoda_log_history, title="A-2: SFT Translator — Training Loss")

In [ ]:
# ── Before/after examples ────────────────────────────────────────────────────
TEST_SENTENCES = [
    "The organization strived for innovation through creating a transparent structure.",
    "Unnatural disasters can cause destruction of property and loss of life.",
    "Knowledge is the key to unlocking human potential.",
]

before_outs = [generate_response(base_model, tokenizer, format_prompt_yoda(s, tokenizer))
               for s in TEST_SENTENCES]
after_outs  = [generate_response(sft_translator_model, tokenizer, format_prompt_yoda(s, tokenizer))
               for s in TEST_SENTENCES]

display_examples(TEST_SENTENCES, before_outs, after_outs, labels=("Base", "SFT"))

### A-3 · Synthetic Yoda-Style QA Dataset

Filter QA pairs from `MuskumPillerum/General-Knowledge` and translate answers using the trained SFT translator. Produces ~500 train / 200 val / 500 test Yoda-style QA pairs.

In [ ]:
qa_raw = load_dataset(QA_DATASET_NAME)
print(f"QA dataset size: {len(qa_raw['train'])} examples")
print("\nSample:", qa_raw["train"][0])

In [ ]:
# ── Multi-tier quality filtering ─────────────────────────────────────────────
N_SUBSAMPLE = 1500   # headroom for post-filter attrition

qa_filter = QAFilter(use_passive_filter=True, min_words=4, max_words=15)

qa_subset = qa_raw["train"].shuffle(seed=SEED).select(range(min(N_SUBSAMPLE, len(qa_raw["train"]))))
clean_pairs = [
    (ex["question"], ex["answer"])
    for ex in qa_subset
    if qa_filter.is_clean(ex["question"], ex["answer"])
]

print(qa_filter.stats())
print(f"\nClean pairs available: {len(clean_pairs)}")

In [ ]:
# ── Translate answers to Yoda style ──────────────────────────────────────────
skip_generating = True   # set False to regenerate

if checkpoint_exists(PATHS["synthetic_qa"] / "train") and skip_generating:
    print("Loading cached synthetic QA dataset...")
    qa_train_yoda = HFDataset.load_from_disk(str(PATHS["synthetic_qa"] / "train"))
    qa_val_yoda   = HFDataset.load_from_disk(str(PATHS["synthetic_qa"] / "val"))
    qa_test_yoda  = HFDataset.load_from_disk(str(PATHS["synthetic_qa"] / "test"))
else:
    N_TRAIN, N_VAL, N_TEST = 500, 200, 500
    selected = clean_pairs[:N_TRAIN + N_VAL + N_TEST]
    questions, answers = zip(*selected)

    # Translate sentence-by-sentence
    all_sents, boundaries = [], []
    for ans in answers:
        sents = split_sentences(ans)
        start = len(all_sents)
        all_sents.extend(sents)
        boundaries.append((start, len(all_sents)))

    prompts     = [format_prompt_yoda(s, tokenizer) for s in all_sents]
    translated  = generate_batch_responses(sft_translator_model, tokenizer, prompts, batch_size=32)
    yoda_answers = [" ".join(translated[s:e]) for s, e in boundaries]

    all_examples = [{"question": q, "original_answer": a, "yoda_answer": y}
                    for q, a, y in zip(questions, answers, yoda_answers)]

    tr = HFDataset.from_list(all_examples[:N_TRAIN])
    va = HFDataset.from_list(all_examples[N_TRAIN:N_TRAIN + N_VAL])
    te = HFDataset.from_list(all_examples[N_TRAIN + N_VAL:])

    PATHS["synthetic_qa"].mkdir(parents=True, exist_ok=True)
    tr.save_to_disk(str(PATHS["synthetic_qa"] / "train"))
    va.save_to_disk(str(PATHS["synthetic_qa"] / "val"))
    te.save_to_disk(str(PATHS["synthetic_qa"] / "test"))
    qa_train_yoda, qa_val_yoda, qa_test_yoda = tr, va, te

print(f"Train: {len(qa_train_yoda)}  Val: {len(qa_val_yoda)}  Test: {len(qa_test_yoda)}")
for ex in qa_train_yoda.select(range(3)):
    print(f"\n  Q: {ex['question']}")
    print(f"  A (original): {ex['original_answer']}")
    print(f"  A (Yoda):     {ex['yoda_answer']}")

### A-4 · SFT: Yoda-Style Question Answering

Fine-tune on the synthetic Yoda QA dataset to teach the model to answer any question in Yoda style.

In [ ]:
# ── Format datasets ──────────────────────────────────────────────────────────
qa_train_fmt = qa_train_yoda.map(lambda ex: format_qa_yoda_example(ex, tokenizer))
qa_val_fmt   = qa_val_yoda.map(lambda ex: format_qa_yoda_example(ex, tokenizer))
print(f"Formatted — Train: {len(qa_train_fmt)}  Val: {len(qa_val_fmt)}")
print("\nSample:", qa_train_fmt[0]["text"][:200])

In [ ]:
# ── Train or load SFT answerer ───────────────────────────────────────────────
skip_training_a4 = True   # set False to retrain

if checkpoint_exists(PATHS["sft_yoda_answ"]) and skip_training_a4:
    print("Loading SFT answerer from checkpoint...")
    sft_yoda_answ_model, tokenizer = load_peft_model(PATHS["sft_yoda_answ"])
    sft_yoda_answ_log = json.load(
        open(PATHS["sft_yoda_answ"] / "trainer_state.json")
    )["log_history"]
else:
    print("Training SFT answerer...")
    free_memory(sft_translator_model)
    model, tokenizer = load_base_model_and_tokenizer(quantize=True, quantize_n_bits=4)

    lora_cfg = get_lora_config(r=16, lora_alpha=32, lora_dropout=0.1)
    trainer  = build_sft_trainer(
        model=model, tokenizer=tokenizer,
        train_dataset=qa_train_fmt, eval_dataset=qa_val_fmt,
        output_dir=PATHS["sft_yoda_answ"],
        num_train_epochs=3, per_device_train_batch_size=4,
        learning_rate=1e-4, lora_config=lora_cfg,
    )
    trainer.train()
    trainer.save_model(str(PATHS["sft_yoda_answ"]))
    sft_yoda_answ_log   = trainer.state.log_history
    sft_yoda_answ_model = trainer.model

In [ ]:
# ── Training curves + validation loss ────────────────────────────────────────
plot_training_loss(sft_yoda_answ_log, title="A-4: SFT Answerer — Training Loss")

eval_entries = [e for e in sft_yoda_answ_log if "eval_loss" in e]
if eval_entries:
    best = min(eval_entries, key=lambda x: x["eval_loss"])
    print(f"Best validation loss: {best['eval_loss']:.4f} at step {best['step']}")

In [ ]:
# ── Before/after examples ────────────────────────────────────────────────────
QA_TEST_PROMPTS = [
    "What is the capital of France?",
    "How does photosynthesis work?",
    "Why do leaves change colour in autumn?",
]

base_model_eval, _ = load_base_model_and_tokenizer(quantize=True, quantize_n_bits=8)
base_model_eval.eval()

before_outs = [generate_response(base_model_eval, tokenizer, format_prompt_qa(q, tokenizer))
               for q in QA_TEST_PROMPTS]
after_outs  = [generate_response(sft_yoda_answ_model, tokenizer, format_prompt_qa(q, tokenizer))
               for q in QA_TEST_PROMPTS]

display_examples(QA_TEST_PROMPTS, before_outs, after_outs, labels=("Base", "SFT-Yoda"))
free_memory(base_model_eval)

---

## Part B · GRPO with Reinforcement Learning from Verifiable Rewards

Five components:

| Step | Description |
|------|-------------|
| **B-1** | Train DistilBERT binary classifier as style reward model |
| **B-2** | Design and validate composite reward function |
| **B-3** | GRPO from SFT checkpoint (correctness + format rewards) |
| **B-4** | GRPO from base model (correctness + format + style rewards) |
| **B-5** | Comparative analysis |

In [ ]:
# ── Verify Part A artifacts ───────────────────────────────────────────────────
print("Checking Part A artifacts:")
for name, path in [
    ("SFT Yoda translator",  PATHS["sft_yoda"]),
    ("SFT Yoda answerer",    PATHS["sft_yoda_answ"]),
    ("Synthetic QA dataset", PATHS["synthetic_qa"]),
]:
    status = "✓" if path.exists() else "✗ MISSING"
    print(f"  {status}  {name}: {path}")

In [ ]:
# ── Load GSM8K reasoning dataset ─────────────────────────────────────────────
reasoning_ds    = load_gsm8k_dataset()
reasoning_train = reasoning_ds["train"]
reasoning_test  = reasoning_ds["test"]
print(f"GSM8K — Train: {len(reasoning_train)}  Test: {len(reasoning_test)}")

### B-1 · Style Reward Model: DistilBERT Classifier

Build a balanced English/Yoda dataset by translating GSM8K answers with the SFT translator, then fine-tune `distilbert-base-uncased` as a binary classifier.

In [ ]:
# ── Build classifier dataset ─────────────────────────────────────────────────
sft_translator_model, tokenizer = load_peft_model(PATHS["sft_yoda"])

cls_train, cls_val, cls_test = build_classifier_dataset(
    reasoning_ds=reasoning_train,
    translator_model=sft_translator_model,
    translator_tokenizer=tokenizer,
    n_samples=1000,
    batch_size=32,
    cache_path=PATHS["classifier_data"],
)
print(f"Train: {len(cls_train)}  Val: {len(cls_val)}  Test: {len(cls_test)}")
free_memory(sft_translator_model)

In [ ]:
# ── Token length distribution of classifier data ────────────────────────────
lengths = [len(tokenizer.encode(ex["text"])) for ex in cls_train]
print(f"Mean: {np.mean(lengths):.0f}  Median: {np.median(lengths):.0f}  "
      f"Max: {max(lengths)}  p95: {np.percentile(lengths, 95):.0f}")

In [ ]:
# ── Train or load classifier ─────────────────────────────────────────────────
skip_training_b1 = True   # set False to retrain

if checkpoint_exists(PATHS["classifier_yoda"]) and skip_training_b1:
    print("Loading classifier from checkpoint...")
    cls_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_NAME)
    cls_model = AutoModelForSequenceClassification.from_pretrained(str(PATHS["classifier_yoda"]))
    cls_log = json.load(open(PATHS["classifier_yoda"] / "trainer_state.json"))["log_history"]
else:
    print("Training DistilBERT classifier...")
    cls_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_NAME)
    cls_model = AutoModelForSequenceClassification.from_pretrained(
        CLASSIFIER_MODEL_NAME, num_labels=2
    )
    trainer = build_classifier_trainer(
        model=cls_model, tokenizer=cls_tokenizer,
        train_ds=cls_train, eval_ds=cls_val,
        output_dir=PATHS["classifier_yoda"],
    )
    trainer.train()
    trainer.save_model(str(PATHS["classifier_yoda"]))
    cls_log   = trainer.state.log_history
    cls_model = trainer.model

In [ ]:
# ── Training curve + test set evaluation ─────────────────────────────────────
plot_training_loss(cls_log, title="B-1: Style Classifier — Training Loss")

cls_results = evaluate_classifier(cls_model, cls_tokenizer, cls_test)
print(f"\nTest Accuracy: {cls_results['accuracy']:.4f}")
print(cls_results["report"])

### B-2 · Composite Reward Function Design

Three components combined linearly:
- **Correctness** — exact numeric match against GSM8K ground truth
- **Format** — presence of `#### <number>` answer marker
- **Style** — DistilBERT classifier probability of Yoda-style output

In [ ]:
# ── Validate rewards on 3 diverse examples ───────────────────────────────────
DEMO_GT = ("Anakin is 'helping' 16 younglings at the Jedi Temple. "
           "3 manage to run away. 4 more escape. #### 9")

DEMO_RESPONSES = [
    {
        "label": "Correct + formatted + Yoda-style",
        "text": "16 younglings, Anakin helps. Run away quickly, 3 do. "
                "Escape 4 more. Remaining younglings, 9 there are. #### 9",
    },
    {
        "label": "Correct + formatted, no Yoda style",
        "text": "There are 16 younglings. 3 run away. 4 escape. "
                "The remaining younglings receive training. #### 9",
    },
    {
        "label": "Wrong answer",
        "text": "Strong with the Force, Anakin is. Counted, the younglings cannot be. #### 7",
    },
]

print(f"{'Example':<45} {'Correctness':>11} {'Format':>8} {'Style':>7} {'Total':>7}")
print("-" * 80)
for ex in DEMO_RESPONSES:
    scores = score_response(
        ex["text"], DEMO_GT,
        use_style=True, classifier_model=cls_model, classifier_tokenizer=cls_tokenizer,
    )
    print(f"{ex['label']:<45} {scores['Correctness']:>11.2f} {scores['Format']:>8.2f} "
          f"{scores['Style']:>7.3f} {scores['Total']:>7.3f}")

### B-3 · GRPO from SFT Checkpoint

Fine-tune the best SFT answerer (A-4) using GRPO with **correctness + format** rewards only (no style). Systematically explores batch size, learning rate, and LR reduction schedules.

In [ ]:
# ── Prepare GRPO training dataset ────────────────────────────────────────────
rl_train, rl_test = prepare_rl_dataset(
    reasoning_ds=reasoning_train, tokenizer=tokenizer,
    n_train=500, n_test=200, max_answer_words=35,
)
print(f"RL dataset — Train: {len(rl_train)}  Test: {len(rl_test)}")
print("\nSample prompt:", rl_train[0]["prompt"][:200])

In [ ]:
# ── Load SFT checkpoint as GRPO starting point ───────────────────────────────
print("Loading SFT answerer as GRPO starting point...")
base_for_merge, tokenizer = load_base_model_and_tokenizer(quantize=False)
sft_for_rl = PeftModel.from_pretrained(base_for_merge, str(PATHS["sft_yoda_answ"]))
sft_for_rl_merged = sft_for_rl.merge_and_unload()
print(f"SFT merged. VRAM: {get_vram_usage()}")

In [ ]:
# ── Debug: SFT model Yoda behaviour on reasoning questions ──────────────────
DEBUG_QUESTIONS = rl_test[:3]["prompt"]
print("SFT model outputs on GSM8K questions (before GRPO):")
for i, prompt in enumerate(DEBUG_QUESTIONS, 1):
    response = generate_response(sft_for_rl_merged, tokenizer, prompt, max_new_tokens=256)
    print(f"\n  [{i}] {response[:200]}...")

In [ ]:
# ── Train or load GRPO (SFT warm start) ─────────────────────────────────────
RL_SFT_OUTPUT = PATHS["rl_yoda_answ_from_sft"]
force_retrain = False

if checkpoint_exists(RL_SFT_OUTPUT) and not force_retrain:
    print(f"Loading B-Q3 checkpoint from {RL_SFT_OUTPUT}...")
    rl_sft_model, tokenizer = load_peft_model(RL_SFT_OUTPUT)
else:
    print("Training GRPO from SFT checkpoint...")
    lora_cfg = get_grpo_lora_config(r=16, lora_alpha=32)
    trainer  = build_grpo_trainer(
        model=sft_for_rl_merged, tokenizer=tokenizer,
        train_dataset=rl_train,
        reward_functions=[correctness_reward, format_reward],
        output_dir=RL_SFT_OUTPUT,
        num_train_epochs=1, per_device_train_batch_size=2,
        learning_rate=5e-6, lora_config=lora_cfg,
    )
    trainer.train()
    trainer.save_model(str(RL_SFT_OUTPUT))
    rl_sft_model = trainer.model

In [ ]:
# ── Compare hyperparameter experiments ───────────────────────────────────────
BQ3_EXP_DIR = BASE_DIR / "bq3_experiments"
if BQ3_EXP_DIR.exists():
    bq3_experiments = plot_experiment_comparison(BQ3_EXP_DIR, section="B-Q3")

In [ ]:
# ── Best model: before vs after analysis ─────────────────────────────────────
BQ3_BEST_PATH = RL_SFT_OUTPUT   # update to best experiment path if using sweep

print("Generating before/after responses...")
N_EVAL = 10
step   = max(1, len(rl_test) // N_EVAL)
eval_idx = list(range(0, len(rl_test), step))[:N_EVAL]

questions_eval   = [rl_test[i]["prompt"]        for i in eval_idx]
ground_truths_bq3 = [rl_test[i]["ground_truth"] for i in eval_idx]

before_outs_bq3 = [generate_response(sft_for_rl_merged, tokenizer, q,
                                      max_new_tokens=512, temperature=0.1)
                    for q in questions_eval]
after_outs_bq3  = [generate_response(rl_sft_model, tokenizer, q,
                                     max_new_tokens=512, temperature=0.1)
                   for q in questions_eval]

bq3_results = compare_before_after(
    questions=[rl_test[i]["prompt"] for i in eval_idx],
    ground_truths=ground_truths_bq3,
    before_responses=before_outs_bq3,
    after_responses=after_outs_bq3,
    before_label="SFT (before RL)",
    after_label="SFT+RL (after)",
    use_style=False,
)

In [ ]:
free_memory(sft_for_rl_merged, rl_sft_model)

### B-4 · GRPO from Base Model with Full Reward

Fine-tune the **base model** (not SFT) using all three reward components: correctness + format + style. Compares cold-start vs Yoda-prompted initialisation strategies.

In [ ]:
# ── Load base model for GRPO cold start ──────────────────────────────────────
print("Loading base model for GRPO cold start...")
base_for_grpo, tokenizer = load_base_model_and_tokenizer(quantize=True, quantize_n_bits=4)
print(f"Model loaded. VRAM: {get_vram_usage()}")

In [ ]:
# ── Build composite reward (correctness + format + style) ────────────────────
cls_model_grpo, cls_tokenizer_grpo = load_style_classifier(str(PATHS["classifier_yoda"]))

def style_reward_fn(prompts, completions, **kwargs):
    return style_reward(
        prompts, completions,
        classifier_model=cls_model_grpo,
        classifier_tokenizer=cls_tokenizer_grpo,
        **kwargs,
    )

reward_functions_full = [correctness_reward, format_reward, style_reward_fn]

In [ ]:
# ── Train or load GRPO (base cold start) ─────────────────────────────────────
RL_BASE_OUTPUT = PATHS["rl_yoda_answ_from_base"]
force_retrain_base = False

if checkpoint_exists(RL_BASE_OUTPUT) and not force_retrain_base:
    print(f"Loading B-Q4 checkpoint from {RL_BASE_OUTPUT}...")
    rl_base_model, tokenizer = load_peft_model(RL_BASE_OUTPUT)
else:
    print("Training GRPO from base model...")
    lora_cfg = get_grpo_lora_config(r=16, lora_alpha=32)
    trainer  = build_grpo_trainer(
        model=base_for_grpo, tokenizer=tokenizer,
        train_dataset=rl_train,
        reward_functions=reward_functions_full,
        output_dir=RL_BASE_OUTPUT,
        num_train_epochs=1, per_device_train_batch_size=2,
        learning_rate=8e-6, lora_config=lora_cfg,
    )
    trainer.train()
    trainer.save_model(str(RL_BASE_OUTPUT))
    rl_base_model = trainer.model

In [ ]:
# ── Compare hyperparameter experiments ───────────────────────────────────────
BQ4_EXP_DIR = BASE_DIR / "bq4_experiments"
if BQ4_EXP_DIR.exists():
    bq4_experiments = plot_experiment_comparison(BQ4_EXP_DIR, section="B-Q4")

In [ ]:
# ── Before vs after analysis ─────────────────────────────────────────────────
base_model_eval, _ = load_base_model_and_tokenizer(quantize=True, quantize_n_bits=4)
base_model_eval.eval()

before_outs_bq4 = [generate_response(base_model_eval, tokenizer, q,
                                      max_new_tokens=512, temperature=0.1)
                   for q in questions_eval]
after_outs_bq4  = [generate_response(rl_base_model, tokenizer, q,
                                     max_new_tokens=512, temperature=0.1)
                   for q in questions_eval]

bq4_results = compare_before_after(
    questions=[rl_test[i]["prompt"] for i in eval_idx],
    ground_truths=ground_truths_bq3,
    before_responses=before_outs_bq4,
    after_responses=after_outs_bq4,
    before_label="Base (before RL)",
    after_label="Base+RL (after)",
    use_style=True,
    classifier_model=cls_model_grpo,
    classifier_tokenizer=cls_tokenizer_grpo,
)

### B-5 · Comparative Analysis

Quantitative comparison of B-Q3 (SFT warm start, 2 rewards) vs B-Q4 (base cold start, 3 rewards) on the held-out GSM8K test set.

In [ ]:
# ── Side-by-side comparison on 6 held-out test examples ─────────────────────
SELECTED_INDICES = [0, 3, 6, 2, 8, 12]

test_questions = [rl_test[i]["prompt"]        for i in SELECTED_INDICES]
test_gts       = [rl_test[i]["ground_truth"]  for i in SELECTED_INDICES]

rl_sft_model, _  = load_peft_model(PATHS["rl_yoda_answ_from_sft"])
rl_base_model, _ = load_peft_model(PATHS["rl_yoda_answ_from_base"])

bq3_outs = [generate_response(rl_sft_model,  tokenizer, q, max_new_tokens=512) for q in test_questions]
bq4_outs = [generate_response(rl_base_model, tokenizer, q, max_new_tokens=512) for q in test_questions]

for i, (q, gt, bq3, bq4) in enumerate(zip(test_questions, test_gts, bq3_outs, bq4_outs), 1):
    gt_num = extract_gsm8k_final_answer(gt)
    s3 = score_response(bq3, gt, use_style=False)
    s4 = score_response(bq4, gt, use_style=True,
                        classifier_model=cls_model_grpo,
                        classifier_tokenizer=cls_tokenizer_grpo)
    print(f"{'=' * 70}")
    print(f"Example {i} | GT = {gt_num}")
    print(f"  [SFT+RL]:  {bq3[:150]}...")
    print(f"  [Base+RL]: {bq4[:150]}...")
    print(f"  Scores    Correctness  Format    Style     Total")
    print(f"  SFT+RL    {s3['Correctness']:>11.2f}  {s3['Format']:>6.2f}    {'N/A':>5}  {s3['Total']:>6.2f}")
    print(f"  Base+RL   {s4['Correctness']:>11.2f}  {s4['Format']:>6.2f}  {s4['Style']:>7.3f}  {s4['Total']:>6.2f}")

In [ ]:
# ── Aggregate comparison table ────────────────────────────────────────────────
def aggregate_scores(responses, ground_truths, use_style, cls_m=None, cls_t=None):
    all_scores = [
        score_response(r, gt, use_style, cls_m, cls_t)
        for r, gt in zip(responses, ground_truths)
    ]
    cols = ["Correctness", "Format"] + (["Style"] if use_style else []) + ["Total"]
    return {c: np.mean([s[c] for s in all_scores]) for c in cols}

agg_bq3 = aggregate_scores(bq3_outs, test_gts, use_style=False)
agg_bq4 = aggregate_scores(bq4_outs, test_gts, use_style=True,
                           cls_m=cls_model_grpo, cls_t=cls_tokenizer_grpo)

print("\nQuantitative Comparison (mean over 6 test examples):")
print(f"{'Metric':<20} {'SFT+RL (B-Q3)':>15} {'Base+RL (B-Q4)':>15}")
print("-" * 52)
for metric in ["Correctness", "Format", "Total"]:
    bq3_val = agg_bq3.get(metric, float("nan"))
    bq4_val = agg_bq4.get(metric, float("nan"))
    print(f"  {metric:<18} {bq3_val:>15.3f} {bq4_val:>15.3f}")
if "Style" in agg_bq4:
    print(f"  {'Style':<18} {'N/A':>15} {agg_bq4['Style']:>15.3f}")

In [ ]:
# ── Visualisation: grouped bar chart ─────────────────────────────────────────
metrics  = ["Correctness", "Format", "Total"]
bq3_vals = [agg_bq3[m] for m in metrics]
bq4_vals = [agg_bq4[m] for m in metrics]

x     = np.arange(len(metrics))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))

bars1 = ax.bar(x - width/2, bq3_vals, width, label="SFT+RL (B-Q3)", color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x + width/2, bq4_vals, width, label="Base+RL (B-Q4)", color="#DD8452", edgecolor="white")

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h, f"{h:.3f}",
                ha="center", va="bottom", fontsize=9, weight="bold")

ax.set_ylabel("Score", weight="bold")
ax.set_title("GRPO Strategy Comparison: SFT Warm Start vs. Base Cold Start",
             weight="bold", pad=12)
ax.set_xticks(x); ax.set_xticklabels(metrics, weight="bold")
ax.legend(); ax.set_ylim(0, 1.15)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("results/comparison.png", dpi=200, bbox_inches="tight")
plt.show()

### Discussion

**Which approach works better overall?**  
Both configurations achieve comparable correctness accuracy. B-Q4 (base + full reward) 
shows superior format compliance, converging to near-perfect adherence to the `#### <n>` 
marker. This reflects the benefit of the style reward providing a dense training signal 
from the first step, anchoring the model's output structure even before reasoning improves.

**Does starting from the SFT checkpoint help or hinder performance?**  
The SFT warm start (B-Q3) provides an advantage in early training stability — the model 
already produces grammatically coherent Yoda-style text, reducing variance in the style 
component. However, the SFT pretraining distribution (general Yoda QA) creates a domain 
shift when faced with GSM8K mathematical reasoning, constraining the model's ability to 
develop novel chain-of-thought strategies.

**Failure modes:**  
- B-Q3: Occasionally produces Yoda syntax but arithmetic errors, suggesting the model 
  optimises style over correctness when they conflict.  
- B-Q4: Cold-start instability in early epochs; the model must simultaneously learn 
  reasoning structure and Yoda syntax from reward signal alone.

**Reward weighting:**  
Equal unit weights (1.0 each) were chosen to avoid biasing the policy towards any single 
component during early training. Future work could explore learned or annealed weightings 
that progressively shift emphasis from format/style to correctness as the model stabilises.